# Using the PhenoMe pipeline with STED-FM models and data
## Synaptic development dataset
---

## 1. Setup

In [1]:
import torch
from phenome import PhenoMe, get_metadata_from_path, load_dinov2_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pheno = PhenoMe(device=device, seed=42)

### Wrap the STED-FM model

In [2]:
from stedfm import get_pretrained_model_v2
from phenome.utils import ModelWrapper

class STEDFMWrapper(ModelWrapper):
    def __init__(self, device):
        # Load the pretrained STED-FM model
        model, cfg = get_pretrained_model_v2(
            name = "mae-lightning-small",
            weights = "MAE_SMALL_STED", 
            in_channels=1, # For single channel images
            as_classifier=True,
            # global_pool = "patch" # Returns the embeddings for all patches
        )

        # Initialize the parent class which moves the model to the device and sets it to eval()
        super().__init__(model=model, device=device)

    def _get_embeddings(self, tensor: torch.Tensor) -> torch.Tensor:
        out = self.model.forward_features(tensor)
        return out

wrapper = STEDFMWrapper(device=device)

in_channels 1
--- mae-lightning-small | (https://s3.valeria.science/flclab-foundation-models/models/mae-small-sted.zip) Downloading from URL and extracting zip ---

Loading state_dict from /Users/renaud/.stedfm/baselines/mae-small_STED/pl_checkpoint-999.pth
--- Loaded model mae-lightning-small with weights MAE_SMALL_STED ---
--- Added linear probe to all frozen blocks ---


### Using DINOv2

In [13]:
wrapper = load_dinov2_model(model_name="dinov2_vits14_reg", device=device)

Using cache found in /Users/renaud/.cache/torch/hub/facebookresearch_dinov2_main
/Users/renaud/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/Users/renaud/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/Users/renaud/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


### Custom transforms for STED dataset

In [3]:
import torchvision.transforms as T

# Transforms for STED-FM model
stedfm_transforms = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224), interpolation=0),
    T.Grayscale(num_output_channels=1),
    T.ToTensor(),
])

# Transforms for STED-FM model
stedfm_transforms = T.Compose([T.ToTensor()])

In [16]:
from skimage import io

img = io.imread("/Users/renaud/datasets/optim-dataset/images/img_3.tif")
print(img.shape)
print(img.min(), img.max())

(224, 224)
0.0 1.0


In [ ]:
from data_handling import SynapticDevelopmentDataset

# Restructure dataset for phenome usage
data_path = "/Users/renaud/datasets/synaptic_development/synaptic_development_dataset_test.tar"

dataset = SynapticDevelopmentDataset(data_path, classes=["DIV5", "DIV14", "DIV16"])

In [38]:
print([dataset[n][1]["label"] for n in range(len(dataset))])

['DIV14', 'DIV14', 'DIV14', 'DIV5', 'DIV5', 'DIV5', 'DIV5', '14', '14', '14', '14', '14', '14', 'DIV14', '14', '14', '14', '14', '14', '14', '14', '14', '14', '14', 'DIV14', '14', '14', '14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', '5', '5', '5', '5', '5', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', '5', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV5', 'DIV5', 'DIV5', 'DIV5', 'DIV5', 'DIV5', 'DIV5', 'DIV5', 'DIV5', 'DIV5', 'DIV14', 'DIV5', 'DIV5', 'DIV5', 'DIV5', 'DIV5', 'DIV5', 'DIV5', 'DIV5', 'DIV5', 'DIV5', 'DIV14', '14', '14', '14', '14', '14', '14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14', 'DIV14

In [44]:
# Iterate through dataset and save individual images, masks and metadata.csv
import os

import numpy as np
import pandas as pd
import tifffile
from tqdm import tqdm

tiles_dir = "/Users/renaud/datasets/synaptic_development_dataset"
tiles_images_dir = os.path.join(tiles_dir, "images")
tiles_metadata_path = os.path.join(tiles_dir, "metadata.csv")

os.makedirs(tiles_images_dir, exist_ok=True)

rows = []
for idx, (image, meta) in enumerate(tqdm(dataset, desc="Saving tiles")):
    stem = (
        f"img_{idx}"
    )

    image = np.asarray(image, dtype=np.float32)
    while image.ndim > 2:
        image = image[0]

    tifffile.imwrite(os.path.join(tiles_images_dir, f"{stem}.tif"), image)
    row_dict = {k: meta.get(k) for k in meta.keys()}
    row_dict["label"] = "DIV" + row_dict["label"] if "DIV" not in row_dict["label"] else row_dict["label"]
    row_dict["filename"] = stem
    rows.append(row_dict)

# 'filename' is the extension-less stem shared by the image and its mask, which is what
# make_dataframe_metadata_fn matches on in pheno.find_files().
tiles_metadata_df = pd.DataFrame(rows)
tiles_metadata_df.to_csv(tiles_metadata_path, index=False)

print(f"Saved {len(tiles_metadata_df)} tiles to {tiles_images_dir}")
print(f"Metadata written to {tiles_metadata_path}")
tiles_metadata_df.head()

Saving tiles: 100%|██████████| 424/424 [00:00<00:00, 1873.74it/s]

Saved 424 tiles to /Users/renaud/datasets/synaptic_development_dataset/images
Metadata written to /Users/renaud/datasets/synaptic_development_dataset/metadata.csv


,label,dataset-idx,batch,dpi,protein,min_value,max_value,filename
0,DIV14,0,26.2,11DPI,PSD95,0.0,53.40554295171458,img_0
1,DIV14,1,26.2,11DPI,PSD95,0.0,53.40554295171458,img_1
2,DIV14,2,26.2,11DPI,PSD95,0.0,53.40554295171458,img_2
3,DIV5,3,26.2,7DPI,PSD95,0.0,34.66184568145915,img_3
4,DIV5,4,26.2,7DPI,PSD95,0.0,34.66184568145915,img_4


### Create dataset (find files)

In [4]:
# csv-based metadata
from phenome import make_dataframe_metadata_fn
from data_handling import minmax_preprocessing_fn
import pandas as pd

images_dir = "/Users/renaud/datasets/synaptic_development_dataset/images"
metadata_dir = "/Users/renaud/datasets/synaptic_development_dataset/metadata.csv"

metadata_df = pd.read_csv(metadata_dir)
metadata_fn = make_dataframe_metadata_fn(metadata_df)
file_df = pheno.find_files(images_dir, mask_dir=None, metadata_fn=metadata_fn)

print(f"Found {len(file_df)} images")



Found 424 files total.
DataFrame: 424 files, 11 columns.


Found 424 images


### Process images

In [5]:
# Process with STED-FM model
checkpoint_path = 'checkpoints/development_dataset/dev_stedfm.h5'

pheno.process_images(wrapper, 
                    force_rgb=False,
                    checkpoint_path=checkpoint_path,
                    preprocessing_fn=None,
                    custom_transformations=stedfm_transforms,
                    l2_normalize_channels=False
                    )
pheno.compute_properties(property_preset="complete", checkpoint_path=checkpoint_path)

Found 424 committed images; skipping 424 (already in checkpoint), 0 to process.
All images already processed. Loading from checkpoint.
Pipeline ready: 424 images available (lazy loading from checkpoints/development_dataset/dev_stedfm.h5).
Properties: 424/424 matched in checkpoint (6 columns).


,image_index,image_path,image_name,intensity_entropy,intensity_max,intensity_mean,intensity_min,intensity_std,sharpness_metric
0,0,/Users/renaud/datasets/synaptic_development_da...,img_0.tif,8.243292,1.000000,0.031608,0.0,0.059912,0.234952
1,1,/Users/renaud/datasets/synaptic_development_da...,img_1.tif,8.411562,1.000000,0.034079,0.0,0.070504,0.236412
2,2,/Users/renaud/datasets/synaptic_development_da...,img_10.tif,5.205038,0.996308,0.022291,0.0,0.054273,0.187864
3,3,/Users/renaud/datasets/synaptic_development_da...,img_100.tif,5.783860,0.656397,0.024994,0.0,0.043851,0.187575
4,4,/Users/renaud/datasets/synaptic_development_da...,img_101.tif,4.499044,0.497729,0.005742,0.0,0.020370,0.236623
...,...,...,...,...,...,...,...,...,...
419,419,/Users/renaud/datasets/synaptic_development_da...,img_95.tif,3.456311,1.000000,0.014210,0.0,0.049900,0.192141
420,420,/Users/renaud/datasets/synaptic_development_da...,img_96.tif,3.316595,1.000000,0.014723,0.0,0.057179,0.220484
421,421,/Users/renaud/datasets/synaptic_development_da...,img_97.tif,6.236501,1.000000,0.028649,0.0,0.065035,0.217906
422,422,/Users/renaud/datasets/synaptic_development_da...,img_98.tif,4.836307,1.000000,0.024304,0.0,0.058243,0.199203


In [15]:
# Process with DINO-v2 model

pheno.process_images(wrapper, 
                    force_rgb=True,
                    channel_mode='combined',
                    checkpoint_path='checkpoints/optim_dataset/optim_dinov2.h5',
                    )
pheno.compute_properties(property_preset="complete")

Processing batches:   0%|          | 0/14 [00:00<?, ?it/s]

Pipeline ready: 438 images available (lazy loading from checkpoints/optim_dataset/optim_dinov2.h5).


Computing properties:   0%|          | 0/438 [00:00<?, ?img/s]

,image_index,image_path,image_name,intensity_min,intensity_max,intensity_std,intensity_mean,sharpness_metric,intensity_entropy
0,0,/Users/renaud/datasets/optim-dataset/images/im...,img_0.tif,0,1,0.079212,0.050424,0.275598,3.755296
1,1,/Users/renaud/datasets/optim-dataset/images/im...,img_1.tif,0,1,0.082970,0.040069,0.264506,2.888038
2,2,/Users/renaud/datasets/optim-dataset/images/im...,img_10.tif,0,1,0.092000,0.054349,0.267108,3.668250
3,3,/Users/renaud/datasets/optim-dataset/images/im...,img_100.tif,0,1,0.064390,0.061956,0.358040,6.042111
4,4,/Users/renaud/datasets/optim-dataset/images/im...,img_101.tif,0,1,0.069090,0.040729,0.232677,2.695590
...,...,...,...,...,...,...,...,...,...
433,433,/Users/renaud/datasets/optim-dataset/images/im...,img_95.tif,0,1,0.072179,0.048363,0.282850,3.839375
434,434,/Users/renaud/datasets/optim-dataset/images/im...,img_96.tif,0,1,0.106250,0.092604,0.309922,4.505223
435,435,/Users/renaud/datasets/optim-dataset/images/im...,img_97.tif,0,1,0.036240,0.006984,0.057381,0.280255
436,436,/Users/renaud/datasets/optim-dataset/images/im...,img_98.tif,0,1,0.063532,0.049130,0.306787,4.324903


## 4. Cluster in embedding space

Clusters are **data-driven** groups in embedding space; they may or may not match your experimental labels. Set `n_clusters` between 2 and the number of images (see [Pipeline API](../../docs/src/content/docs/advanced/api/pipeline.md)). After clustering, labels are stored under metadata key `cluster`.

In [7]:
pheno.compute_clustering(n_clusters=2, clustering_method="kmeans", dbscan_eps=0.5)
print(pheno.get_available_property_keys())
print(pheno.get_available_metadata_keys())

pheno.plot_tsne(color_by="label")
pheno.plot_tsne(color_by="cluster")

Clustering: PCA extracted 100 components from 384 dimensions before k-means.
Clustering (kmeans): 2 clusters on 424 images (source=embeddings).


['intensity_entropy', 'intensity_max', 'intensity_mean', 'intensity_min', 'intensity_std', 'sharpness_metric']
['batch', 'cluster', 'dataset-idx', 'dpi', 'filename', 'id', 'label', 'mask_path', 'max_value', 'min_value', 'protein']


## Interactive Exploration

The pipeline provides an interactive dashboard using Plotly and ipywidgets. This allows you to explore the embedding space, highligh specific groups, and inspect individual images by clicking on dots.

Note: This requires a Jupyter environment with `ipywidgets` and `plotly` installed.


In [51]:
# Launch interactive explorer (if in Jupyter)
explorer = pheno.create_interactive_explorer()

In [42]:
print(pheno.get_embeddings())

[[ 0.4940979  -0.15642206  0.16755281 ... -0.08746281 -0.10659971
   0.23227245]
 [ 0.7412068   0.02284756  0.02199288 ... -0.32124972 -0.21679543
   0.02871149]
 [ 0.4910155  -0.0863172   0.04681274 ... -0.15222144 -0.22366065
   0.14079086]
 ...
 [ 1.4690845  -0.09965071  0.05177866 ... -0.35084644 -0.10247457
   0.23225403]
 [ 0.4877159  -0.07567067  0.07629384 ... -0.06640187 -0.14330094
   0.09132167]
 [ 0.91226655 -0.07476225  0.24811381 ... -0.43762866 -0.24789537
   0.11671644]]
